In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Cargar el dataset (Igual que tenías)
df = pd.read_csv('../data/processed/consumo_granada_modelo.csv')

# 2. Separar Features y Target (Igual que tenías)
X = df.drop('consumption_kwh', axis=1)
y = df['consumption_kwh']

# 3. División Train/Test (Igual que tenías)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# 4. Instanciar el modelo
# 'random_state' fija la aleatoriedad para que el resultado sea siempre el mismo
gb_model = GradientBoostingRegressor(random_state=42,
                                     n_estimators=300,        # <-- CAMBIO CLAVE
                                     learning_rate=0.05,      # <-- CAMBIO CLAVE
                                     max_depth=5,             # <-- CAMBIO CLAVE
                                     min_samples_leaf=10      # <-- CAMBIO CLAVE
                                    )

print("Entrenando y validando Gradient Boosting (esto puede tardar un poco)...")


Entrenando y validando Gradient Boosting (esto puede tardar un poco)...


In [11]:
# 5. Validación Cruzada
scoring_metrics = ['neg_mean_absolute_error', 'neg_root_mean_squared_error']

cv_results = cross_validate(
    gb_model, 
    X_train, 
    y_train, 
    cv=5,                 
    scoring=scoring_metrics, 
    n_jobs=-1                  
)

# 6. Procesar los resultados
mae_promedio = -1 * cv_results['test_neg_mean_absolute_error'].mean()
rmse_promedio = -1 * cv_results['test_neg_root_mean_squared_error'].mean()

print("\nResultados de la Validación Cruzada (Promedio de 5 pruebas):")
print(f"MAE Promedio:  {mae_promedio:.2f} kWh")
print(f"RMSE Promedio: {rmse_promedio:.2f} kWh")


Resultados de la Validación Cruzada (Promedio de 5 pruebas):
MAE Promedio:  171.75 kWh
RMSE Promedio: 259.43 kWh


In [12]:
# 7. Comparación con el Objetivo del Proyecto
objetivo_mae = 218.58
objetivo_rmse = 342.82

print("\n--- Auditoría del Proyecto ---")
if mae_promedio <= objetivo_mae:
    print(f"✅ ¡OBJETIVO CUMPLIDO! Tu error ({mae_promedio:.2f}) es menor que la referencia ({objetivo_mae}).")
else:
    print(f"⚠️ Aún no superas la referencia. Diferencia: {mae_promedio - objetivo_mae:.2f} kWh.")


--- Auditoría del Proyecto ---
✅ ¡OBJETIVO CUMPLIDO! Tu error (171.75) es menor que la referencia (218.58).


In [13]:
# 8. Entrenar el modelo final
gb_model.fit(X_train, y_train)
print("\nModelo final entrenado y listo para guardar.")


Modelo final entrenado y listo para guardar.


In [ ]:
# ============================================
# CELDA: GUARDAR SOLO EL MODELO DE GRADIENT BOOSTING
# ============================================
import os
import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# 1. Crear el directorio de modelos si no existe
os.makedirs("../data/models", exist_ok=True)

# 2. Definir la ruta y guardar SOLO el modelo entrenado
model_path = "../data/models/gradient_boosting_model.joblib"
joblib.dump(gb_model, model_path)

print(f"💾 Modelo Gradient Boosting guardado en: {model_path}")
print(f"   Tamaño: {os.path.getsize(model_path) / (1024*1024):.2f} MB")

# 3. Validar con el test set
print(f"\n🧪 Evaluando en test set ({len(X_test)} muestras)...")
y_pred = gb_model.predict(X_test)

mae_test = mean_absolute_error(y_test, y_pred)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"\n📊 Métricas en Test (Gradient Boosting):")
print(f"   MAE:  {mae_test:.2f} kWh")
print(f"   RMSE: {rmse_test:.2f} kWh")

print("\n✅ Proceso de guardado y validación completado.")

💾 Modelo Gradient Boosting guardado en: ../data/models/gradient_boosting_model.joblib
   Tamaño: 1.37 MB

🧪 Evaluando en test set (486866 muestras)...

📊 Métricas en Test (Gradient Boosting):
   MAE:  171.29 kWh
   RMSE: 260.30 kWh

✅ Proceso de guardado y validación completado.
